# Part 2: ASL to English Translation using T5 text-to-text Transformer

## 1. Importing Dependencies

In [68]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup
import torch
from torch.utils.data import DataLoader, Dataset
import random
import re
import PyPDF2
import pandas as pd

## 2. Dataset Creation:

## The PDF file in this folder named 'sign language phrases.pdf' is read to create the database
## Some sentences were added separately as well

In [70]:
# Initialize lists to hold extracted parts
outside_brackets = []
inside_brackets = []

# Function to extract and split text
def process_text(text):
    # Define the pattern to match any ASL gloss (quoted or unquoted) followed by a translation in parentheses
    pattern = r'^\d+\.\s*(["]?)(.*?)\1\s*\((.*?)\)'
    
    # Split text by line
    lines = text.splitlines()
    
    for line in lines:
        # Check if the line matches the desired format
        match = re.search(pattern, line)
        if match:
            # Append matched parts to lists
            outside_brackets.append(match.group(2).strip())  # ASL gloss
            inside_brackets.append(match.group(3).strip())   # Translation

# Open the PDF file
with open('sign language phrases.pdf', 'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    
    # Extract text from each page and process it
    pdf_text = ""
    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        pdf_text += page.extract_text()

# Process the extracted text to split phrases and meanings
process_text(pdf_text)

outside_brackets = [item.lower() for item in outside_brackets]
inside_brackets = [item.lower() for item in inside_brackets]

In [71]:
data = [

    ("FRIEND CALL ME YESTERDAY", "My friend called me yesterday."),
    ("SISTER HELP ME WITH HOMEWORK", "My sister helped me with homework."),
    ("BROTHER PLAY FOOTBALL", "My brother played football."),
    ("MOM COOK DINNER", "My mom cooked dinner."),
    ("DAD FIX CAR", "My dad fixed the car."),
    ("COUSIN VISIT TOMORROW", "My cousin is visiting tomorrow."),
    ("TEACHER EXPLAIN HOMEWORK", "The teacher explained the homework."),
    ("FRIEND MEET PARK", "I met my friend at the park."),
    ("UNCLE WORK HARD", "My uncle works hard."),
    ("AUNT HELP WITH GARDEN", "My aunt helped with the garden."),
    ("LIBRARY, BOOKS READ", "I read books in the library."),
    ("GO MARKET BUY FOOD", "I go to the market to buy food."),
    ("PARK PLAY SOCCER", "I play soccer at the park."),
    ("SCHOOL CLASSROOM STUDY", "I study in the classroom at school."),
    ("HOME CLEANING NOW", "I am cleaning the house now."),
    ("MALL SHOPPING, CLOTHES BUY", "I am shopping at the mall to buy clothes."),
    ("CLOSE RESTAURANT, EAT DINNER", "I eat dinner at the close restaurant."),
    ("GARDEN, FLOWERS PLANT", "I plant flowers in the garden."),
    ("BUS STOP WAIT", "I wait at the bus stop."),
    ("OFFICE, COMPUTER WORK", "I work on the computer at the office."),
    ("YESTERDAY I GO TO PARK", "I went to the park yesterday."),
    ("TOMORROW I WILL MEET FRIEND", "I will meet my friend tomorrow."),
    ("LUNCHTIME, EAT SANDWICH", "I eat a sandwich at lunchtime."),
    ("MORNING I EXERCISE", "I exercise in the morning."),
    ("LAST WEEK I VISITED FAMILY", "I visited my family last week."),
    ("FEEL HAPPY TODAY", "I feel happy today."),
    ("SAD I FEEL", "I feel sad."),
    ("EXCITED I AM", "I am excited."),
    ("TIRED TODAY I FEEL", "I feel tired today."),
    ("SURPRISED I AM", "I am surprised."),
    ("ANGRY I FEEL", "I feel angry."),
    ("SCARED I AM", "I am scared."),
    ("WORRIED I AM", "I am worried."),
    ("RELAXED I FEEL", "I feel relaxed."),
    ("HUNGRY I AM", "I am hungry."),
    ("PARK, PLAY SOCCER", "I play soccer at the park."),
    ("SCHOOL LUNCHTIME, EAT", "I eat during lunchtime at school."),
    ("CUP OF COFFEE, DRINK ME", "I drink a cup of coffee."),
    ("DINNER EAT AT HOME", "I eat dinner at home."),
    ("PIZZA ORDER TONIGHT", "I will order pizza tonight."),
    ("FEEL THIRSTY", "I feel thirsty."),
    ("STORMY IT IS", "It is stormy."),
    ("COLD TODAY", "Today is cold."),
    ("SUNNY WEATHER", "The weather is sunny."),
    ("RAINY IT IS", "It is rainy."),
    ("HOT WEATHER", "The weather is hot."),
    ("WINDY IT IS", "It is windy."),
    ("SNOWING NOW", "It is snowing now."),
    ("HEALTHY I FEEL", "I feel healthy."),
    ("SICK I AM", "I am sick."),
    ("EXERCISE EVERYDAY", "I exercise every day."),
    ("HEADACHE I HAVE", "I have a headache."),
    ("COUGHING I AM", "I am coughing."),
    ("STOMACH PAIN I HAVE", "I have stomach pain."),
    ("RUNNING I AM", "I am running."),
    ("SLEEPY I FEEL", "I feel sleepy."),
    ("RUNNING FAST", "I am running fast."),
    ("COOK MEAL, YUM", "I cook a meal, and it’s yummy."),
    ("EAT PIZZA TONIGHT", "I will eat pizza tonight."),
    ("DRINK WATER", "I drink water."),
    ("EAT BREAKFAST", "I eat breakfast."),
    ("FEEL FULL NOW", "I feel full now."),
    ("COFFEE DRINK IN MORNING", "I drink coffee in the morning."),
    ("FAVORITE COLOR BLUE", "My favorite color is blue."),
    ("YELLOW COLOR BRIGHT", "Yellow is a bright color."),
    ("RED COLOR LOVE", "I love the color red."),
    ("BLACK COLOR BEAUTIFUL", "Black is a beautiful color."),
    ("WHITE COLOR PURE", "White is a pure color."),
    ("PINK COLOR PRETTY", "Pink is a pretty color."),
    ("GREEN COLOR CALM", "Green is a calming color."),
    ("BROWN COLOR WARM", "Brown is a warm color."),
    ("ORANGE COLOR BRIGHT", "Orange is a bright color."),
    ("PURPLE COLOR DEEP", "Purple is a deep color."),
    ("MARRIAGE HAPPY BOTH US", "We are both happy in our marriage."),
    ("LOVE YOU", "I love you."),
    ("BOYFRIEND CALL ME", "My boyfriend called me."),
    ("PARTNER DATE NIGHT", "My partner and I went on a date night."),
    ("WIFE BEAUTIFUL, LOVE ME", "My wife is beautiful, and I love her."),
    ("HUSBAND COOK, SURPRISE ME", "My husband cooked and surprised me."),
    ("GIRLFRIEND HAPPY SEE ME", "My girlfriend is happy to see me."),
    ("RELATIONSHIP GOOD", "The relationship is good."),
    ("LOVE TRUST NEED BOTH", "Love and trust are both needed."),
    ("FRIEND SUPPORT ME ALWAYS", "My friend always supports me."),
    ("ENGAGED NOW", "I’m engaged now."),
    ("BOYFRIEND GIFT GIVE ME", "My boyfriend gave me a gift."),
    ("RELATIONSHIP STRONG", "Our relationship is strong."),
    ("PARTNER TRUST ME", "My partner trusts me."),
    ("LOVE IMPORTANT RELATIONSHIP", "Love is important in a relationship."),
    ("SISTER HELP ME", "My sister helps me."),
    ("HUSBAND FIX CAR", "My husband fixed the car."),
    ("BROTHER FIX BIKE", "My brother fixed the bike."),
    ("COUSIN PLAY GAME", "My cousin plays a game."),
    ("MOM COOK SOUP", "My mom cooked soup."),
    ("DAD DRIVE CAR", "My dad drives the car."),
    ("FRIEND VISIT ME", "My friend visited me."),
    ("WIFE LOVE ME", "My wife loves me."),
    ("HUSBAND SUPPORT ME", "My husband supports me."),
    ("AUNT VISIT NOW", "My aunt is visiting now."),
    ("BROTHER PLAY VIDEO GAME", "My brother is playing video games."),
    ("SISTER COOK LUNCH", "My sister is cooking lunch."),
    ("COUSIN WALK DOG", "My cousin walks the dog."),
    ("DAD FIX COMPUTER", "My dad fixed the computer."),
    ("MOM CLEAN HOUSE", "My mom cleaned the house."),
    ("FRIEND HELP ME STUDY", "My friend helps me study."),
    ("FAMILY EAT TOGETHER", "My family eats together."),
    ("HUSBAND DOING LAUNDRY", "My husband is doing the laundry."),
    ("PARTNER TALK TO ME", "My partner is talking to me."),
    ("FRIEND GIVE GIFT", "My friend gave me a gift."),
    ("WIFE HELP WITH PROJECT", "My wife helped with the project."),
    ("DAD FIX CHAIR", "My dad fixed the chair."),
    ("SISTER VISIT FAMILY", "My sister visited the family."),
    ("BROTHER COOK BREAKFAST", "My brother cooked breakfast."),
    ("COUSIN HELP MOVE", "My cousin helped move."),
    ("AUNT TRAVEL TO PARIS", "My aunt traveled to Paris."),
    ("MOM LAUNDRY WASH", "My mom washed the laundry."),
    ("TEACHER HELP STUDENT", "The teacher helped the student."),
    ("BOYFRIEND LAUGH WITH ME", "My boyfriend laughed with me."),
    ("FRIEND SHARE FOOD", "My friend shared food with me."),
    ("PARTNER LOVE ME FOREVER", "My partner will love me forever."),
    ("SISTER GIVE ADVICE", "My sister gave me advice."),
    ("HUSBAND SURPRISE ME", "My husband surprised me."),
    ("WIFE LOVE ME ALWAYS", "My wife will always love me."),
    ("FRIEND RELAX WITH ME", "My friend relaxed with me."),
    ("PARENTS SUPPORT ME", "My parents support me."),
    ("SIBLINGS ALWAYS PLAY TOGETHER", "My siblings always play together."),
    ("MOTHER CARE FOR ME", "My mother cares for me."),
    ("FATHER HELP ME LEARN", "My father helped me learn."),
    ("RELATIONSHIP STRONGER EVERYDAY", "Our relationship grows stronger every day."),
    ("LOVE MAKE US STRONG", "Love makes us strong."),
    ("HUSBAND TRUSTS ME", "My husband trusts me.")

]


print(len(data))

130


In [72]:
asl_gloss = [item[0].lower() for item in data]
english_translations = [item[1].lower() for item in data]
outside_brackets+=asl_gloss
inside_brackets+=english_translations

In [73]:
print(len(outside_brackets))

623


In [78]:
visualize_data = pd.DataFrame({'Asl Gloss': outside_brackets, 'English Translation': inside_brackets})
visualize_data.style

,Asl Gloss,English Translation
0,you name?,what is your name?
1,deaf you?,are you deaf?
2,student you?,are you a student?
3,your teacher name 'huh'?,what is your teacher's name?
4,you understand he/she?,do you understand him/her?
5,who he/she?,who is he/she?
6,"again, name you?",what is your name again?
7,"""s/he student s/he?"" [point at teacher]",is she or he a student?
8,"""this your?"" [point at any object]",is this yours?
9,"""where "" [spell the name of a person in the room]",where is [person]?


## Identify the top n words based on frequency of appearance

In [80]:
tuples_list = list(zip(outside_brackets, inside_brackets))

In [82]:
first_elements = list(list(zip(*tuples_list))[0]) 
print(first_elements)

['you  name?', 'deaf  you?', 'student  you?', "your  teacher name 'huh'?", 'you  understand  he/she?', 'who he/she?', 'again, name  you?', '"s/he student  s/he?" [point at teacher]', '"this  your?" [point at any object]', '"where  " [spell the name of a person in the room]', 'nice  meet -you', 'hearing  you?', 'asl  teacher  you?', 'you learn sign, where?', 'you learn sign, why  ?', 'how  sign  t-h-a-n-k-s?', '"student,  he/she  ?" [point  at other  student]', 'they  learn sign?', 'your  teacher, who?', 'you like learn sign?', 'hey,  name  you?', 'you married?', 'children  you?', 'you  sister how -many?', "your mom name 'huh'?", 'your dad deaf?', 'you work where?', 'you live where ?', 'this his/hers? [point at any object]', 'how  you sign  w-e? [spell "we"]', 'you name b -o-b, you?', 'you divorced  you?', 'how -many  brother  you?', 'your sister single?', 'your  dad  name,  spell  slow.', 'you-meet  my brother  you?', 'you live where?', 'you  have  sister?', 'this  your? [point at any 

In [52]:
from collections import Counter

def get_most_common_words(sentences, top_n=200):
    # Step 1: Combine all sentences into one string
    text = " ".join(sentences)
    
    # Step 2: Tokenize (split) the text into words and remove non-alphanumeric characters
    words = re.findall(r'\b\w+\b', text.lower())  # Convert to lowercase and extract words
    
    # Step 3: Count the frequency of each word
    word_counts = Counter(words)
    
    # Step 4: Get the top 'n' most common words
    most_common = word_counts.most_common(top_n)
    
    return most_common

top_100_words = get_most_common_words(first_elements, top_n=200)

# Display the results
print("Top 100 Most Common Words:")
for word, frequency in top_100_words:
    print(f"{word}: {frequency}")


Top 100 Most Common Words:
you: 376
your: 123
i: 95
like: 83
what: 81
how: 76
have: 45
many: 35
sign: 29
name: 23
who: 22
think: 22
go: 21
color: 21
eat: 21
want: 19
t: 17
feel: 17
class: 17
where: 16
a: 16
s: 15
school: 15
kind: 15
before: 15
why: 14
house: 14
live: 13
favorite: 13
do: 12
car: 12
time: 12
work: 11
need: 11
to: 11
when: 11
equal: 11
deaf: 10
teacher: 10
this: 10
learn: 10
he: 9
dad: 9
e: 9
family: 9
book: 9
me: 9
for: 9
she: 8
my: 8
dog: 8
take: 8
up: 8
one: 8
at: 7
any: 7
here: 7
help: 7
doctor: 7
play: 7
past: 7
cook: 7
know: 7
new: 7
people: 7
student: 6
spell: 6
asl: 6
h: 6
sister: 6
friend: 6
can: 6
home: 6
now: 6
tomorrow: 6
year: 6
prefer: 6
cat: 6
read: 6
day: 6
huh: 5
point: 5
in: 5
meet: 5
n: 5
mom: 5
city: 5
good: 5
l: 5
come: 5
movie: 5
with: 5
old: 5
apple: 5
food: 5
pizza: 5
don: 5
which: 5
finish: 5
c: 5
yesterday: 5
next: 5
test: 5
sleep: 5
often: 5
night: 5
self: 5
enjoy: 5
hair: 5
strong: 5
study: 5
of: 4
room: 4
hearing: 4
k: 4
b: 4
o: 4
brother: 4
b

## 3. T5 Model and Training

In [84]:
# Initialize the T5 tokenizer and model
model_name = "t5-small"  # Use "t5-base" or "t5-large" for better performance
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Sample ASL-English data pairs (you should expand this with a larger dataset)
data = tuples_list

# Data Augmentation: Duplicate examples with slight variations if you don't have a large dataset
#augmented_data = data * 100  # Duplicate data 100 times for this example
augmented_data = data

# Create a custom dataset class
class ASLDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        asl, eng = self.data[idx]
        input_text = f"translate ASL to English: {asl}"
        target_text = eng
        return input_text, target_text

# Initialize dataset and dataloader
dataset = ASLDataset(augmented_data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Training parameters
epochs = 30
learning_rate = 5e-5

# Optimizer and scheduler setup
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
num_training_steps = epochs * len(dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [86]:
# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_texts, target_texts = batch

        # Tokenize input and target texts
        input_encodings = tokenizer(input_texts, padding=True, truncation=True, return_tensors="pt")
        target_encodings = tokenizer(target_texts, padding=True, truncation=True, return_tensors="pt")

        # Forward pass
        outputs = model(input_ids=input_encodings["input_ids"], 
                        attention_mask=input_encodings["attention_mask"], 
                        labels=target_encodings["input_ids"])

        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    average_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{epochs} Loss: {average_loss}")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1/30 Loss: 2.9011845191319785
Epoch 2/30 Loss: 1.4667545748062623
Epoch 3/30 Loss: 1.024307594085351
Epoch 4/30 Loss: 0.8640048156181971
Epoch 5/30 Loss: 0.8390865394702325
Epoch 6/30 Loss: 0.7171939687851148
Epoch 7/30 Loss: 0.668081209827692
Epoch 8/30 Loss: 0.6315613144483322
Epoch 9/30 Loss: 0.5925159314886118
Epoch 10/30 Loss: 0.5671648818712968
Epoch 11/30 Loss: 0.5445856543687674
Epoch 12/30 Loss: 0.5194245206239896
Epoch 13/30 Loss: 0.5079519704748423
Epoch 14/30 Loss: 0.4646137142792726
Epoch 15/30 Loss: 0.47114696888587415
Epoch 16/30 Loss: 0.46074026421858716
Epoch 17/30 Loss: 0.446065397025683
Epoch 18/30 Loss: 0.42835988333592045
Epoch 19/30 Loss: 0.4114629232730621
Epoch 20/30 Loss: 0.4206709130070148
Epoch 21/30 Loss: 0.3933675312078916
Epoch 22/30 Loss: 0.3836218893336944
Epoch 23/30 Loss: 0.3841170816658399
Epoch 24/30 Loss: 0.37691470904228014
Epoch 25/30 Loss: 0.3765079743013932
Epoch 26/30 Loss: 0.3695542541069862
Epoch 27/30 Loss: 0.3664493948603288
Epoch 28/

## 4. Testing

In [112]:
def is_question(sentence):
    # List of common question words
    question_words = ['what', 'how', 'why', 'where', 'who', 'when', 'is', 'are', 'can', 'could', 'will', 'do', 'does']
    
    # Split the sentence into words
    words = sentence.split()
    
    # Check if the list contains any question word or if the last word is 'you'
    if any(word.lower() in question_words for word in words) or words[-1].lower() == 'you' or words[-1].lower() == 'name':
        return True
    return False

def add_question_mark(sentence):
    # Check if the sentence is a question
    if is_question(sentence):
        # Ensure the sentence ends with a question mark
        if not sentence.endswith('?'):
            sentence += '?'
    return sentence

In [122]:
# Test the model with a new ASL input
def translate_asl(asl_sentence):
    model.eval()
    input_ids = tokenizer(f"translate ASL to English: {asl_sentence}", return_tensors="pt").input_ids
    outputs = model.generate(input_ids)
    translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_sentence

# Example translation
test_sentence = "student you"
updated_test_sentence=add_question_mark(test_sentence)
print(updated_test_sentence)
translation = translate_asl(updated_test_sentence)
print("ASL Gloss:", updated_test_sentence)
print("English Translation:", translation)

student you?
ASL Gloss: student you?
English Translation: are you a student?


C:\Users\Siddhant\anaconda3\Lib\site-packages\transformers\generation\utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


## 5. Saving the trained Model

In [124]:
output_dir = "fine_tuned_model"  # Specify the directory to save the model
model.save_pretrained(output_dir)

In [126]:
tokenizer.save_pretrained(output_dir)

('fine_tuned_model\\tokenizer_config.json',
 'fine_tuned_model\\special_tokens_map.json',
 'fine_tuned_model\\spiece.model',
 'fine_tuned_model\\added_tokens.json')

## 6. Using the trained model after saving

In [54]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("fine_tuned_model")
tokenizer = T5Tokenizer.from_pretrained("fine_tuned_model")

In [66]:
# Test the model with a new ASL input
def translate_asl(asl_sentence):
    model.eval()
    input_ids = tokenizer(f"translate ASL to English: {asl_sentence}", return_tensors="pt").input_ids
    outputs = model.generate(input_ids)
    translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_sentence

# Example translation
test_sentence = "hello student you?"
translation = translate_asl(test_sentence)
print("ASL Gloss:", test_sentence)
print("English Translation:", translation)

C:\Users\Siddhant\anaconda3\Lib\site-packages\transformers\generation\utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


ASL Gloss: hello student you?
English Translation: do you have a student?
